# Object-Oriented Programming

Notebook for **Chapter 8** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch08.html

Tip: change a number, predict the output, then run the cell again.

## 8.1 Classes, Instances and Attributes

In [ ]:
class Counter:
    total_created = 0          # class attribute
    def __init__(self, name):
        self.name = name       # instance attribute
        self.count = 0
        Counter.total_created += 1
    def hit(self):
        self.count += 1
        return self.count

a = Counter("login")
b = Counter("search")
a.hit(); a.hit(); b.hit()
print(a.name, a.count, b.name, b.count)
print(Counter.total_created, a.total_created)
a.total_created = 99          # creates an instance attribute, shadows class one
print(a.total_created, b.total_created, Counter.total_created)
print(a.__dict__)
print(type(a), type(Counter))

login 2 search 1
2 2
99 2 2
{'name': 'login', 'count': 2, 'total_created': 99}
<class '__main__.Counter'> <class 'type'>


In [ ]:
class Bad:
    tags = []
    def add(self, t): self.tags.append(t)
x, y = Bad(), Bad()
x.add("a"); y.add("b")
print(x.tags, y.tags, x.tags is y.tags)
class Good:
    def __init__(self): self.tags = []
    def add(self, t): self.tags.append(t)
x, y = Good(), Good()
x.add("a"); y.add("b")
print(x.tags, y.tags)

['a', 'b'] ['a', 'b'] True
['a'] ['b']


In [ ]:
class A:
    def hit(): return 1       # forgot self
A().hit()

A.hit() takes 0 positional arguments but 1 was given


In [ ]:
class B:
    def __init__(self, n): self.n = n
B()

B.__init__() missing 1 required positional argument: 'n'


## 8.2 Dunder (Special) Methods

In [ ]:
from functools import total_ordering
@total_ordering
class Money:
    def __init__(self, rupees, paise=0):
        self.paise = rupees*100 + paise
    def __repr__(self): return f"Money({self.paise//100}, {self.paise%100})"
    def __str__(self): return f"Rs {self.paise/100:.2f}"
    def __eq__(self, o): return isinstance(o, Money) and self.paise == o.paise
    def __lt__(self, o): return self.paise < o.paise
    def __hash__(self): return hash(self.paise)
    def __add__(self, o): return Money(0, self.paise + o.paise)
    def __bool__(self): return self.paise != 0

m1, m2 = Money(10, 50), Money(3, 75)
print(repr(m1), str(m1), m1 + m2)
print(m1 > m2, m1 == Money(10,50), m1 >= m2)
print(sorted([m1, m2, Money(1)]))
print({m1, Money(10,50)})
print(bool(Money(0)))
print([m1, m2])

Money(10, 50) Rs 10.50 Rs 14.25
True True True
[Money(1, 0), Money(3, 75), Money(10, 50)]
{Money(10, 50)}
False
[Money(10, 50), Money(3, 75)]


In [ ]:
class Playlist:
    def __init__(self, *songs): self._s = list(songs)
    def __len__(self): return len(self._s)
    def __getitem__(self, i): return self._s[i]
    def __iter__(self): return iter(self._s)
    def __contains__(self, x): return x in self._s
p = Playlist("a","b","c")
print(len(p), p[0], p[-1], p[0:2], "b" in p, list(p))

class Timer:
    def __init__(self, f): self.f = f; self.calls = 0
    def __call__(self, *a):
        self.calls += 1
        return self.f(*a)
sq = Timer(lambda x: x*x)
print(sq(4), sq(5), sq.calls)

3 a c ['a', 'b'] True ['a', 'b', 'c']
16 25 2


In [ ]:
class P:
    def __init__(self, x): self.x = x
    def __eq__(self, o): return self.x == o.x
{P(1)}

cannot use 'P' as a set element (unhashable type: 'P')


## 8.3 Inheritance,super(), MRO and Mixins

In [ ]:
class Animal:
    def __init__(self, name): self.name = name
    def speak(self): return "..."
    def intro(self): return f"{self.name} says {self.speak()}"
class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed
    def speak(self): return "Woof"
d = Dog("Rex", "Lab")
print(d.intro(), isinstance(d, Animal), issubclass(Dog, Animal))

Rex says Woof True True


In [ ]:
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B>" + super().who()
class C(A):
    def who(self): return "C>" + super().who()
class D(B, C):
    def who(self): return "D>" + super().who()
print(D().who())
print([k.__name__ for k in D.__mro__])

D>B>C>A
['D', 'B', 'C', 'A', 'object']


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
class JsonMixin:
    def to_json(self):
        import json
        return json.dumps(self.__dict__)
class LogMixin:
    def log(self, msg): print(f"[{type(self).__name__}] {msg}")
class User(JsonMixin, LogMixin):
    def __init__(self, n, a): self.name=n; self.age=a
u = User("Asha", 30)
u.log(u.to_json())
try:
    class X(A, B): pass
except TypeError as e: print("TypeError:", e)

[User] {"name": "Asha", "age": 30}
TypeError: Cannot create a consistent method resolution order (MRO) for bases A, B


## 8.4 Abstract Base Classes and Protocols

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from abc import ABC, abstractmethod

class Exporter(ABC):
    @abstractmethod
    def export(self, rows): ...
    def run(self, rows):
        return f"{len(rows)} rows -> {self.export(rows)}"
class CsvExporter(Exporter):
    def export(self, rows): return "\n".join(",".join(map(str,r)) for r in rows)
try:
    Exporter()
except TypeError as e: print("TypeError:", e)
class Half(Exporter): pass
try: Half()
except TypeError as e: print("TypeError:", e)
print(CsvExporter().run([(1,2),(3,4)]))

TypeError: Can't instantiate abstract class Exporter without an implementation for abstract method 'export'
TypeError: Can't instantiate abstract class Half without an implementation for abstract method 'export'
2 rows -> 1,2
3,4


In [ ]:
from typing import Protocol, runtime_checkable

class Closeable(Protocol):
    def close(self) -> None: ...
class FileLike:
    def close(self): print("closed file-like")
def shutdown(x: Closeable): x.close()
shutdown(FileLike())    # no inheritance needed

@runtime_checkable
class Sized2(Protocol):
    def __len__(self) -> int: ...
print(isinstance([1], Sized2), isinstance(5, Sized2))

closed file-like
True False


## 8.5 Properties,@classmethodand@staticmethod

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
class Temperature:
    def __init__(self, c): self.celsius = c
    @property
    def celsius(self): return self._c
    @celsius.setter
    def celsius(self, v):
        if v < -273.15: raise ValueError("below absolute zero")
        self._c = v
    @property
    def fahrenheit(self): return self._c*9/5+32
t = Temperature(100)
print(t.celsius, t.fahrenheit)
t.celsius = 0
print(t.fahrenheit)
try: t.celsius = -300
except ValueError as e: print("ValueError:", e)
try: t.fahrenheit = 5
except AttributeError as e: print("AttributeError:", e)

100 212.0
32.0
ValueError: below absolute zero
AttributeError: property 'fahrenheit' of 'Temperature' object has no setter


In [ ]:
class Server:
    def __init__(self, host, port): self.host, self.port = host, port
    @classmethod
    def from_url(cls, url):
        host, port = url.split(":")
        return cls(host, int(port))
    @staticmethod
    def valid_port(p): return 0 < p < 65536
    def __repr__(self): return f"Server({self.host!r}, {self.port})"
print(Server.from_url("db.local:5432"), Server.valid_port(70000))

Server('db.local', 5432) False


## 8.6 Dataclasses, NamedTuple, Enum and__slots__

In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from dataclasses import dataclass, field, asdict, replace

@dataclass
class Request:
    url: str
    latency_ms: float = 0.0
    tags: list = field(default_factory=list)
    def __post_init__(self):
        if self.latency_ms < 0: raise ValueError("negative latency")
r = Request("/login", 120.5)
print(r, r == Request("/login", 120.5))
print(asdict(r), replace(r, latency_ms=99))
try: Request("/x", -1)
except ValueError as e: print("ValueError:", e)

Request(url='/login', latency_ms=120.5, tags=[]) True
{'url': '/login', 'latency_ms': 120.5, 'tags': []} Request(url='/login', latency_ms=99, tags=[])
ValueError: negative latency


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from dataclasses import dataclass
from typing import NamedTuple
from enum import Enum, auto

@dataclass(frozen=True, order=True)
class Point:
    x: int
    y: int
p = Point(1,2)
try: p.x = 5
except Exception as e: print(type(e).__name__, e)
print(sorted([Point(2,1), Point(1,9)]), {p: "ok"})

class Pt(NamedTuple):
    x: int
    y: int
q = Pt(3,4)
x, y = q
print(q, q.x, q[1], x+y, q._asdict())

class Status(Enum):
    OK = 200
    NOT_FOUND = 404
    ERR = auto()
print(Status.OK, Status(404), Status["ERR"].value, Status.OK.name)
print([s.name for s in Status])

FrozenInstanceError cannot assign to field 'x'
[Point(x=1, y=9), Point(x=2, y=1)] {Point(x=1, y=2): 'ok'}
Pt(x=3, y=4) 3 4 7 {'x': 3, 'y': 4}
Status.OK Status.NOT_FOUND 405 OK
['OK', 'NOT_FOUND', 'ERR']


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
import sys
class Plain:
    def __init__(self,a,b): self.a=a; self.b=b
class Slim:
    __slots__ = ("a","b")
    def __init__(self,a,b): self.a=a; self.b=b
pl, sl = Plain(1,2), Slim(1,2)
print(sys.getsizeof(pl), sys.getsizeof(pl.__dict__), sys.getsizeof(sl))
try: sl.c = 3
except AttributeError as e: print("AttributeError:", e)

48 296 48
AttributeError: 'Slim' object has no attribute 'c' and no __dict__ for setting new attributes


## 8.7 Composition, the Four Pillars and SOLID

In [ ]:
class Engine:
    def start(self): return "engine on"
class Car:
    def __init__(self, engine): self.engine = engine
    def start(self): return "car: " + self.engine.start()
class FakeEngine:
    def start(self): return "fake"
print(Car(Engine()).start(), "|", Car(FakeEngine()).start())

car: engine on | car: fake


In [ ]:
class Bird:
    def fly(self): return "flying"
class Penguin(Bird):
    def fly(self): raise NotImplementedError("penguins cannot fly")
def migrate(birds): return [b.fly() for b in birds]
try: migrate([Bird(), Penguin()])
except NotImplementedError as e: print("LSP broken:", e)

LSP broken: penguins cannot fly


## 8.8 Design Patterns You Must Recognise

In [ ]:
from typing import Callable

def no_discount(t): return t
def festival(t): return t*0.9
def bulk(t): return t*0.8 if t > 1000 else t
STRATEGIES: dict[str, Callable[[float], float]] = {"none": no_discount, "festival": festival, "bulk": bulk}
def price(total, rule): return STRATEGIES[rule](total)
print(price(1500, "festival"), price(1500, "bulk"), price(500, "bulk"))

class Config:
    _inst = None
    def __new__(cls):
        if cls._inst is None:
            cls._inst = super().__new__(cls)
            cls._inst.values = {}
        return cls._inst
c1, c2 = Config(), Config()
c1.values["env"] = "prod"
print(c1 is c2, c2.values)
import logging
print(logging.getLogger("a") is logging.getLogger("a"))

1350.0 1200.0 500
True {'env': 'prod'}
True


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from abc import ABC, abstractmethod
class Reader(ABC):
    @abstractmethod
    def read(self, text): ...
class CsvReader(Reader):
    def read(self, text): return [l.split(",") for l in text.splitlines()]
class PipeReader(Reader):
    def read(self, text): return [l.split("|") for l in text.splitlines()]
def make_reader(kind):
    readers = {"csv": CsvReader, "pipe": PipeReader}
    try: return readers[kind]()
    except KeyError: raise ValueError(f"unknown reader {kind!r}") from None
print(make_reader("pipe").read("a|b\nc|d"))
try: make_reader("xml")
except ValueError as e: print("ValueError:", e)

[['a', 'b'], ['c', 'd']]
ValueError: unknown reader 'xml'


In [ ]:
class QueryBuilder:
    def __init__(self, table): self.table=table; self._where=[]; self._limit=None; self._params=[]
    def where(self, cond, *params): self._where.append(cond); self._params += params; return self
    def limit(self, n): self._limit = n; return self
    def build(self):
        sql = f"SELECT * FROM {self.table}"
        if self._where: sql += " WHERE " + " AND ".join(self._where)
        if self._limit is not None: sql += f" LIMIT {int(self._limit)}"
        return sql, tuple(self._params)
print(QueryBuilder("orders").where("status = ?", "paid").where("amount > ?", 500).limit(10).build())

('SELECT * FROM orders WHERE status = ? AND amount > ? LIMIT 10', ('paid', 500))


In [ ]:
class EventBus:
    def __init__(self): self._subs = {}
    def subscribe(self, event, fn): self._subs.setdefault(event, []).append(fn)
    def publish(self, event, data):
        for fn in self._subs.get(event, []): fn(data)
bus = EventBus()
bus.subscribe("order", lambda d: print("email sent for", d))
bus.subscribe("order", lambda d: print("stock reduced by", d["qty"]))
bus.publish("order", {"id": 7, "qty": 2})

class Notifier:
    def send(self, m): return m
class Upper:
    def __init__(self, inner): self.inner = inner
    def send(self, m): return self.inner.send(m.upper())
class Stars:
    def __init__(self, inner): self.inner = inner
    def send(self, m): return "*** " + self.inner.send(m)
print(Stars(Upper(Notifier())).send("alert"))

class OldLogger:
    def write_line(self, s): print("OLD:", s)
class LoggerAdapter:
    def __init__(self, old): self.old = old
    def info(self, msg): self.old.write_line(f"INFO {msg}")
LoggerAdapter(OldLogger()).info("started")

email sent for {'id': 7, 'qty': 2}
stock reduced by 2
*** ALERT
OLD: INFO started


In [ ]:
import sqlite3
from dataclasses import dataclass
from typing import Protocol

@dataclass
class User:
    id: int
    name: str

class UserRepo(Protocol):
    def add(self, name: str) -> User: ...
    def get(self, id: int) -> User | None: ...

class SqliteUserRepo:
    def __init__(self, conn):
        self.conn = conn
        conn.execute("CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY, name TEXT)")
    def add(self, name):
        cur = self.conn.execute("INSERT INTO users(name) VALUES (?)", (name,))
        return User(cur.lastrowid, name)
    def get(self, id):
        row = self.conn.execute("SELECT id, name FROM users WHERE id = ?", (id,)).fetchone()
        return User(*row) if row else None

class InMemoryUserRepo:
    def __init__(self): self.d = {}
    def add(self, name):
        u = User(len(self.d)+1, name); self.d[u.id] = u; return u
    def get(self, id): return self.d.get(id)

class SignupService:
    def __init__(self, repo: UserRepo):
        self.repo = repo
    def signup(self, name):
        if not name.strip(): raise ValueError("empty name")
        return self.repo.add(name.strip())

for repo in (InMemoryUserRepo(), SqliteUserRepo(sqlite3.connect(":memory:"))):
    svc = SignupService(repo)
    u = svc.signup("  Meera ")
    print(type(repo).__name__, u, repo.get(u.id), repo.get(99))

InMemoryUserRepo User(id=1, name='Meera') User(id=1, name='Meera') None
SqliteUserRepo User(id=1, name='Meera') User(id=1, name='Meera') None


## 8.9 Under the Hood: Descriptors, Subclass Hooks, Hashing and Caching

Every feature in this chapter, from `@property` to `super()`, is built on a small set of rules in Python's data model. Learning those rules lets you build validation, plugin systems and caches that feel native.

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = "_" + name
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.name)
    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name[1:]} must be positive, got {value}")
        setattr(obj, self.name, value)
class Item:
    price = Positive()
    qty = Positive()
    def __init__(self, price, qty):
        self.price = price
        self.qty = qty
it = Item(2.5, 4)
print(it.price * it.qty, it.__dict__)
try:
    Item(-1, 3)
except ValueError as e:
    print(e)
print(type(Item.price).__name__)

10.0 {'_price': 2.5, '_qty': 4}
price must be positive, got -1
Positive


In [ ]:
class Exporter:
    registry = {}
    def __init_subclass__(cls, fmt=None, **kw):
        super().__init_subclass__(**kw)
        if fmt:
            Exporter.registry[fmt] = cls
class CsvExporter(Exporter, fmt="csv"):
    def run(self): return "a,b"
class JsonExporter(Exporter, fmt="json"):
    def run(self): return '{"a": 1}'
print(sorted(Exporter.registry))
print(Exporter.registry["json"]().run())

['csv', 'json']
{"a": 1}


In [ ]:
class Bad:
    def __init__(self, k): self.k = k
    def __eq__(self, o): return self.k == o.k
try:
    {Bad(1)}
except TypeError as e:
    print(e)
class Good:
    def __init__(self, k): self.k = k
    def __eq__(self, o): return isinstance(o, Good) and self.k == o.k
    def __hash__(self): return hash(self.k)
print(len({Good(1), Good(1), Good(2)}))
class Mutable:
    def __init__(self, k): self.k = k
    def __eq__(self, o): return self.k == o.k
    def __hash__(self): return hash(self.k)
m = Mutable(1); s = {m}
m.k = 2
print(m in s, Mutable(1) in s, len(s))

cannot use 'Bad' as a set element (unhashable type: 'Bad')
2
False False 1


In [ ]:
from functools import cached_property
class Report:
    def __init__(self, rows): self.rows = rows
    @cached_property
    def total(self):
        print("computing")
        return sum(self.rows)
r = Report([1, 2, 3])
print(r.total, r.total)
del r.total
r.rows.append(4)
print(r.total)

computing
6 6
computing
10


In [ ]:
# This cell is meant to fail: read the error message, that is the lesson.
from dataclasses import dataclass, field, replace, fields
import sys
@dataclass(frozen=True, slots=True)
class Order:
    id: int
    items: list = field(default_factory=list)
    total: float = field(init=False, default=0.0)
    def __post_init__(self):
        if self.id <= 0:
            raise ValueError("id must be positive")
        object.__setattr__(self, "total", float(len(self.items)))
o = Order(1, ["a", "b"])
print(o)
try:
    o.id = 5
except Exception as e:
    print(type(e).__name__)
print(replace(o, id=2).id, hasattr(o, "__dict__"))
@dataclass
class Plain:
    id: int
try:
    @dataclass
    class Oops:
        xs: list = []
except ValueError as e:
    print(e)
print([f.name for f in fields(Order)])

Order(id=1, items=['a', 'b'], total=2.0)
FrozenInstanceError
2 False
mutable default <class 'list'> for field xs is not allowed: use default_factory
['id', 'items', 'total']


In [ ]:
class A:
    def hi(self): return "A"
class B(A):
    def hi(self): return "B>" + super().hi()
class C(A):
    def hi(self): return "C>" + super().hi()
class D(B, C):
    def hi(self): return "D>" + super().hi()
print(D().hi())
print([k.__name__ for k in D.__mro__])

D>B>C>A
['D', 'B', 'C', 'A', 'object']


## Chapter recap

## Practice set

Problem 8.9.1 (easy) What prints?

In [ ]:
class C:
    n = 0
c1, c2 = C(), C()
c1.n = 5
print(c2.n)

In [ ]:
class Cfg:
    def __init__(self, d={}): self.d = d
a, b = Cfg(), Cfg()
a.d["k"] = 1
print(b.d)

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch08.html